In [0]:
%run ./07-medallion-approach

In [0]:
class medallionApproachTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.invoices_bz")
        spark.sql("drop table if exists spark_stream.default.invoice_line_items")

        dbutils.fs.rm(f"{self.checkpoint_dir}/invoices_bz",True)
        dbutils.fs.rm(f"{self.checkpoint_dir}/invoice_line_items",True)

        dbutils.fs.rm(f"{self.base_data_dir}/data/invoices_archive", True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/invoices", True)
        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/invoices")
        print("Done")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/invoices/invoices-{itr}.json", f"{self.base_data_dir}/data/invoices/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql("select count(*) from spark_stream.default.invoice_line_items").collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")  


    
    def validateArchive(self):
        print("\tStarting Archive Validation...", end='')

        archive_path = (f"{self.base_data_dir}/data/invoices_archive")
        expected_files = {"invoices-1.json","invoices-2.json","invoices-3.json"}

        # Recursively collect archived filenames
        actual_files = set()

        def collectFiles(path):
            for f in dbutils.fs.ls(path):
                if f.isDir():
                    collectFiles(f.path)
                else:
                    actual_files.add(f.name)

        collectFiles(archive_path)

        # Validate expected files
        missing_files = expected_files - actual_files

        assert not missing_files, (f"Archive validation failed! Missing files: {missing_files}")

        print("Done")
        print(f"Successfully archived: {sorted(actual_files)}")
  
    def runTests(self):
        self.cleanTests()

        bzStream = Bronze()
        slStream = Silver()

        print("\nTesting first iteration of invoice stream...")
        self.ingestData(1)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()

        slQuery = slStream.process()
        slQuery.awaitTermination()

        self.assertResult(1253)
        print("Validation passed.\n")

        print("Testing second iteration of invoice stream...")
        self.ingestData(2)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()

        slQuery = slStream.process()
        slQuery.awaitTermination()

        self.assertResult(2510)
        print("Validation passed.\n")

        print("Testing third iteration of invoice stream...")
        self.ingestData(3)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()

        slQuery = slStream.process()
        slQuery.awaitTermination()

        self.assertResult(3994)
        print("Validation passed.\n")

        print("Testing fourth iteration of invoice stream...")
        self.ingestData(4)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()

        slQuery = slStream.process()
        slQuery.awaitTermination()

        self.assertResult(4016)
        print("Validation passed.\n")

        print("Validating Archive...")

        print("\nValidating Archive...")
        self.validateArchive()


In [0]:
maTS = medallionApproachTestSuite()
maTS.runTests()